# CS 243 - Homework 2

In [21]:
import ipywidgets as widgets
from ipywidgets import interact
import matplotlib.pyplot as plt
import pandas as pd
from adjustText import adjust_text

box_raw = pd.read_csv("team_box.zip")
rk = pd.read_csv("rankings.csv")

MIN_GAMES = 15


def update_plot(season=2026):
  box = box_raw[box_raw.season == season].drop_duplicates(
      ["game_id", "team_id"]
  )

  cols = [
      "field_goals_attempted",
      "offensive_rebounds",
      "turnovers",
      "free_throws_attempted",
      "team_score",
  ]
  opp = box[["game_id", "team_id"] + cols].rename(
      columns={"team_id": "opponent_team_id", **{c: f"opp_{c}" for c in cols}}
  )
  g = box.merge(opp, on=["game_id", "opponent_team_id"])

  poss = (
      lambda p: p("field_goals_attempted")
      - p("offensive_rebounds")
      + p("turnovers")
      + 0.475 * p("free_throws_attempted")
  )
  g["poss"] = 0.5 * (poss(lambda c: g[c]) + poss(lambda c: g["opp_" + c]))
  g = g[g.poss > 0]

  t = (
      g.groupby(["team_id", "team_short_display_name"])
      .agg(
          games=("game_id", "count"),
          pts=("team_score", "sum"),
          opp_pts=("opp_team_score", "sum"),
          poss=("poss", "sum"),
      )
      .reset_index()
  )
  t = t[t.games >= MIN_GAMES]
  t["off"] = 100 * t.pts / t.poss
  t["def"] = 100 * t.opp_pts / t.poss

  ap = rk[(rk.name == "AP Top 25") & rk.current.between(1, 25)][
      ["team_id", "current"]
  ]
  t = t.merge(ap, on="team_id", how="left")
  top, rest = t[t.current.notna()], t[t.current.isna()]

  fig, ax = plt.subplots(figsize=(12, 7))
  ax.scatter(rest.off, rest["def"], s=15, color="#ccc", label="Other teams")

  shade = 0.9 - 0.65 * (top.current - 1) / 24
  ax.scatter(
      top.off,
      top["def"],
      s=170,
      c=plt.cm.Blues(shade),
      edgecolor="#08306b",
      linewidth=0.6,
      zorder=3,
      label="AP Top 25",
  )

  texts = []
  for (_, r), sh in zip(top.iterrows(), shade):
    ax.text(
        r.off,
        r["def"],
        str(int(r.current)),
        ha="center",
        va="center",
        fontsize=7,
        weight="bold",
        color="white" if sh > 0.55 else "black",
        zorder=4,
    )
    if r.current <= 10:
      texts.append(
          ax.text(r.off, r["def"], r.team_short_display_name, fontsize=8)
      )

  adjust_text(
      texts,
      ax=ax,
      force_points=(2.5, 3.0),
      force_text=(1.5, 2.0),
      expand=(2.0, 2.0),
      arrowprops=dict(arrowstyle="-", color="#888", lw=0.5),
  )

  ax.invert_yaxis()
  ax.set_xlabel("Offense: points scored per 100 possessions")
  ax.set_ylabel("Defense: points allowed per 100 possessions (inverted)")
  ax.set_title(
      f"Offense vs. Defense, AP Top 25 Highlighted (Season: {season})"
  )

  x_margin = (t.off.max() - t.off.min()) * 0.08
  ax.set_xlim(t.off.min() - x_margin, t.off.max() + x_margin * 2.5)

  ax.legend(loc="upper left")
  plt.tight_layout()
  plt.show()


interact(
    update_plot, season=widgets.IntSlider(min=2020, max=2026, step=1, value=2026)
)

FileNotFoundError: [Errno 2] No such file or directory: 'team_box.zip'